In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
orders_df = spark.table(
    "ecommerce_lakehouse.silver.orders_clean"
)

customers_df = spark.table(
    "ecommerce_lakehouse.silver.customers_clean"
)

order_items_df = spark.table(
    "ecommerce_lakehouse.silver.order_items_clean"
)

payments_df = spark.table(
    "ecommerce_lakehouse.silver.payments_clean"
)

products_df = spark.table(
    "ecommerce_lakehouse.silver.products_clean"
)

In [0]:
fact_orders_df = (
    orders_df.alias("o")

    .join(
        customers_df.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    )

    .join(
        order_items_df.alias("oi"),
        col("o.order_id") == col("oi.order_id"),
        "left"
    )

    .join(
        payments_df.alias("p"),
        col("o.order_id") == col("p.order_id"),
        "left"
    )

    .join(
        products_df.alias("pr"),
        col("oi.product_id") == col("pr.product_id"),
        "left"
    )
)

In [0]:
display(fact_orders_df)

In [0]:
fact_orders_df = fact_orders_df.select(

    # order info
    col("o.order_id"),
    col("o.order_status"),
    col("o.purchase_date"),
    col("o.purchase_year"),
    col("o.purchase_month"),
    col("o.delivery_days"),

    # customer info
    col("c.customer_unique_id"),
    col("c.customer_city"),
    col("c.customer_state"),

    # product info
    col("pr.product_id"),
    col("pr.product_category_name"),
    col("pr.weight_category"),

    # payment info
    col("p.payment_type"),
    col("p.payment_installments"),

    # financial metrics
    col("oi.price"),
    col("oi.freight_value"),
    col("oi.total_item_value"),
    col("p.payment_value")
)

In [0]:
fact_orders_df = fact_orders_df.withColumn(
    "is_large_order",
    when(col("payment_value") > 500, True)
    .otherwise(False)
)

In [0]:
(
    fact_orders_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            "ecommerce_lakehouse.gold.fact_orders"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.gold.fact_orders
LIMIT 20;